In [ ]:
%pip install google-genai pymupdf numpy pandas langchain langchain_text_splitters

Note: you may need to restart the kernel to use updated packages.


# Extracción de datos

In [ ]:
from google import genai
from time import sleep
import numpy as np
import pandas as pd
import requests
import pymupdf
import os
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


In [2]:
if not os.path.isdir("data"):   # Comprobar si no existe el directorio
    os.mkdir("data")  # Crear el directorio

if not os.path.isdir("pdf"):
    os.mkdir("pdf")

if not os.path.isdir("texto"):
    os.mkdir("texto")

os.chdir("data")   # Acceder al directorio data
os.chdir("pdf")   # Acceder al directorio pdf

In [ ]:
# Descargar PDFs y guardar en data/pdf

urls = (
    "https://www.utb.edu.co/wp-content/uploads/2026/10/Reglamento-Estudiantil-2026.pdf",
    "https://www.utb.edu.co/wp-content/uploads/2026/03/Reglamento-Ruta-de-autonomia.pdf",
    "https://www.utb.edu.co/wp-content/uploads/2025/09/reglamento-del-CJ-26-09-25.pdf",
    "https://www.utb.edu.co/wp-content/uploads/2021/04/reglamento_estudio_de_television_y_laboratorio_de_posproduccion_0_0.pdf",
    "https://www.utb.edu.co/wp-content/uploads/2021/04/reglamento_laboratorio_de_fotografia_0.pdf",
    "https://www.utb.edu.co/wp-content/uploads/2021/04/reglamento_laboratorio_de_sonido_0.pdf",
    "https://www.utb.edu.co/wp-content/uploads/2021/04/reglamento_sala_multimedia.pdf",
    "https://www.utb.edu.co/wp-content/uploads/2021/06/REGLAMENTO-DE-USO-DE-LOS-LABORATORIOS-DE-PSICOLOGIA.pdf"
)

for url in urls:
    nombre = url.split("/")[-1]
    
    try:
        resp = requests.get(url)
        with open(nombre, "wb") as f:
            f.write(resp.content)

        print(f"Documento guardado: {os.getcwd()}/{nombre}.")
    except Exception as e:
        print(f"Ha ocurrido un error con {nombre}:", e)
        break
    
    sleep(2)

Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/pdf/REGLAMENTO-ORGANICO-ACTUALIZADO-2021-Publicacion.pdf.
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/pdf/Reglamento-Estudiantil-2026.pdf.
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/pdf/Reglamento-Ruta-de-autonomia.pdf.
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/pdf/estatutos_generales_actualizado_2018.doc.pdf.
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/pdf/reglamento-del-CJ-26-09-25.pdf.
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/pdf/reglamento_estudio_de_television_y_laboratorio_de_posproduccion_0_0.pdf.
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/pdf/reglamento_laboratorio_de_fotografia_0.pdf.
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/pdf/reglamento_laborator

# Linea base: 50 preguntas sin recuperación

In [ ]:
response = client.models.generate_content(
 model="gemini-3.5-flash-lite",
 contents="Hola"
)

print(response.text)

¡Hola! ¿En qué puedo ayudarte hoy?


# División en chunks

In [56]:
def extraer(texto, titulos):
    contenidos = []
    end = 0    # indice hasta donde copiar
    texto = "      " + texto  # Para evitar problemas al calcular end

    for t in titulos:
        if t in ("INFORMACIÓN PREVIA", ""): continue # Primer "título"
        end = texto.find(t,3)            # calculo indice hasta donde copiar
        contenidos.append(texto[:end]) # agrego el contenido a la lista
        texto = texto[end:]            # ahora el texto comenzará donde terminó

    contenidos.append(texto)   # agrego lo restante

    longitud = len(titulos)
    if longitud != len(set(titulos)): # si los titulos estan repetidos
        titulos = [str(i) for i in range(longitud)] # asignar un numero unico para que no se sobreescriba
    
    return dict(zip(titulos, contenidos))


def insertar_chunks_en_db(reglamento, name, titulos_cap, regex_articulos):
    global df
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,   # Máximo 500 palabras por bloque
        chunk_overlap=50, # Solapamiento de 50 palabras entre bloques
        length_function=lambda x: len(x.split()),  # Cuenta palabras separadas por espacios
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    # cuando se va a extraer, a la lista de capitulos se le añade uno llamado "INFORMACIÓN PREVIA"
    # para capturar el contenido anterior al primer capitulo y para que no
    # se descuadren los capitulos
    capitulos = extraer(reglamento, ["INFORMACIÓN PREVIA"] + titulos_cap)

    # si no hay informacion previa, quitar la clave-valor
    if capitulos["INFORMACIÓN PREVIA"].strip() == "":
        del capitulos["INFORMACIÓN PREVIA"]

    # Extraer articulos y poner todo en la base de datos
    for titulo in capitulos:
        # eliminar el titulo del capitulo del texto del capitulo y hacerle strip
        content_cap = capitulos[titulo].replace(titulo, '').strip()
        lista_art = regex_articulos.findall(content_cap) # lista de articulos

        # si el texto del capitulo no tiene articulos sino texto continuo,
        # dividir en chunks de 500 palabras
        if len(lista_art) == 0:
            docs =  [Document(content_cap)] # convertir a Document de langchain
            splits = text_splitter.split_documents(docs)  # llamar al splitter
            
            for split in splits: # agregar cada chunk
                chunk = split.page_content   # contenido del chunk
                if chunk.strip() == "": continue
                
                nuevafila = pd.DataFrame({
                    "chunk": [chunk],
                    "capitulo": [titulo],
                    "documento": [name]
                })
    
                df = pd.concat([df, nuevafila], ignore_index=True)

            continue # no ejecutar lo de abajo

        # si el texto del capítulo sí se puede dividir en artículos,
        # extraer diccionario con los articulos. A lista_art se le agrega un
        # titulo vacío "" para que no se descuadre el diccionario
        articulos = extraer(content_cap, [""] + lista_art)

        # por cada articulo, agregar un chunk
        for name_chunk, chunk in zip(articulos.keys(), articulos.values()):
            if chunk.strip() == "": continue
            nuevafila = pd.DataFrame({
                "chunk": [chunk],
                "capitulo": [titulo],
                "documento": [name]
            })

            df = pd.concat([df, nuevafila], ignore_index=True)

In [44]:
reglamentos = []       # textos de los pdfs
titulos_capitulos = [] # titulos de los capitulos de los pdfs
patrones = []          # patrones para encontrar los chunks o articulos
nombres = []           # nombres de los pdfs


for name in ["Reglamento-Ruta-de-autonomia.pdf", "reglamento-del-CJ-26-09-25.pdf"]:
    reglamento = ""   # aqui irá el texto completo del reglamento actual
    doc = pymupdf.open(name)  # leer el pdf

    for i, pagina in enumerate(doc):    # iterar por paginas
        texto = pagina.get_text().strip()

        reemplazo = "REGLAMENTO DE LA RUTA DE AUTONOMÍA Y ÉXITO PROFESIONALPARA ESTUDIANTES DE PREGRADO PRESENCIAL\n" + str(i+1) if ((i+1) % 2 != 0) else ""
        reemplazo = "UNIVERSIDAD TECNOLÓGICA DE BOLÍVAR\n" + str(i+1) if ((i+1) % 2 == 0) else reemplazo
        texto = texto.replace(reemplazo, "")  # quitar texto que se repite en todas las paginas
        texto = texto.replace("CAPITULO III", "CAPÍTULO III")  # corregir una I sin tilde
        texto += "\n"
        reglamento += texto

    reglamentos.append(reglamento)
    nombres.append(name[:-4])

    p = re.compile(r"(?s)(CAP[ÍI]TULO\s*.*?)\s*Art[íi]culo")
    titulos_capitulos.append( p.findall(reglamento) )
    patrones.append(re.compile(r"Artículo .*?\."))

In [45]:
reglamento = ""
name = "Reglamento-Estudiantil-2026.pdf"
doc = pymupdf.open(name)
reemplazo = """Reglamento Estudiantil
Aprobado en Acta No. 03 del 17 de mayo de 2024 del Consejo Académico Resolución 
No. 01 de 2024 y Modificado por la Resolución del Consejo Académico No. 01 de 2026"""

for i, pagina in enumerate(doc):
    if (i+1) in [2,3,4]:   # omitir paginas de CONTENIDO
        continue
    texto = pagina.get_text().strip()
    impar = str(i+1) + '\n' if ((i+1) % 2 != 0) else ""
    par = '\n' + str(i+1) if ((i+1) % 2 == 0) else ""
    texto = texto.replace(f"{impar}{reemplazo}{par}", "")  # quitar texto que se repite en todas las paginas
    texto += "\n"
    reglamento += texto

reglamentos.append(reglamento)
nombres.append(name[:-4])


p = re.compile(r"(?m)^[IVXLM]+\.(?!\s+(?:LENGUAS EXTRANJERAS|RECLAMOS))\s[\s\S]*?(?=\nARTÍCULO|\nA\.)")
titulos_capitulos.append( ["PRESENTACIÓN", "GLOSARIO"] + p.findall(reglamento) )
patrones.append( re.compile(r"ARTÍCULO .*?:") )

In [46]:
nombres_de_otros_reglamentos = [
    "REGLAMENTO-DE-USO-DE-LOS-LABORATORIOS-DE-PSICOLOGIA.pdf",
    "reglamento_estudio_de_television_y_laboratorio_de_posproduccion_0_0.pdf",
    "reglamento_laboratorio_de_fotografia_0.pdf",
    "reglamento_laboratorio_de_sonido_0.pdf",
    "reglamento_sala_multimedia.pdf"
]

cap_de_otros_reglamentos = [
    ["REGLAMENTO DE USO DE LOS LABORATORIOS DE PSICOLOGÍA \nLABPSIC"],
    
    ["REGLAMENTO ESTUDIO DE TELEVISIÓN Y LABORATORIO DE\nPOSPRODUCCIÓN.",
     "DISPOSICIONES GENERALES.", "Utilización de equipos al interior y fuera de la Universidad\nTecnológica de Bolívar.",
     "Son causales para perder el turno de prácticas\ny uso de equipos."],
    
    ["REGLAMENTO LABORATORIO DE FOTOGRAFÍA.", "DISPOSICIONES GENERALES.", 
     "Utilización de equipos al interior de la Universidad\nTecnológica de Bolívar",
     "Utilización de equipos fuera de la Universidad Tecnológica de\nBolívar.",
     "Son causales para perder el turno de\nprácticas."],
    
    ["REGLAMENTO LABORATORIO DE SONIDO", "DISPOSICIONES GENERALES.", "Son causales para perder el turno de\nprácticas."],
    
    ["REGLAMENTO SALA DE REDACCIÓN.", "DISPOSICIONES GENERALES."]
]

for name, titulos_cap in zip(nombres_de_otros_reglamentos, cap_de_otros_reglamentos):
    reglamento = ""
    doc = pymupdf.open(name)
    pagina_del_articulo = {}
    vineta = 1

    for i, pagina in enumerate(doc):
        texto = pagina.get_text().strip()
        texto = texto.replace("\n", "-- ")  # Reemplazar viñeta
        texto = texto.replace("", "-- ") # Reemplazar viñeta
        texto += "\n"
        reglamento += texto
    
    reglamentos.append(reglamento)
    nombres.append(name[:-4])
    if name == "REGLAMENTO-DE-USO-DE-LOS-LABORATORIOS-DE-PSICOLOGIA.pdf":
        patrones.append(re.compile(r"\d+\."))
    else:
        patrones.append(re.compile("-- "))
    titulos_capitulos.append(titulos_cap)

In [78]:
df = pd.DataFrame({"chunk": [],
                   "capitulo": [], 
                   "documento": [],
                   "embeddings": []})

for i in range(len(reglamentos)):
    insertar_chunks_en_db(reglamentos[i], nombres[i], titulos_capitulos[i], patrones[i])

df["n_palabras"] = (
    df["chunk"].str.split().str.len()
)

df.to_html("../db.html")
df.to_csv("../db.csv")
df

,chunk,capitulo,documento,embeddings,n_palabras
0,Reglamento de la \nRuta de autonomía y \néxito...,INFORMACIÓN PREVIA,Reglamento-Ruta-de-autonomia,NaN,173
1,Artículo 1º.- Objeto.\nEl presente reglamento ...,CAPÍTULO I\nDISPOSICIONES GENERALES,Reglamento-Ruta-de-autonomia,NaN,45
2,Artículo 2º.- Naturaleza.\nLa Ruta de Autonomí...,CAPÍTULO I\nDISPOSICIONES GENERALES,Reglamento-Ruta-de-autonomia,NaN,43
3,Artículo 3º.- Ámbito de aplicación.\nEl presen...,CAPÍTULO I\nDISPOSICIONES GENERALES,Reglamento-Ruta-de-autonomia,NaN,104
4,Artículo 4º.- Objetivos.\nSon objetivos de la ...,CAPÍTULO I\nDISPOSICIONES GENERALES,Reglamento-Ruta-de-autonomia,NaN,50
...,...,...,...,...,...
330,"-- Procurar por el buen uso de las sillas, mes...",DISPOSICIONES GENERALES.,reglamento_sala_multimedia,NaN,49
331,-- Ningún estudiante puede modificar conexione...,DISPOSICIONES GENERALES.,reglamento_sala_multimedia,NaN,22
332,-- La temperatura del aire acondicionado será ...,DISPOSICIONES GENERALES.,reglamento_sala_multimedia,NaN,17
333,-- En caso de presentar inconvenientes con los...,DISPOSICIONES GENERALES.,reglamento_sala_multimedia,NaN,28


Guardado en `data/db.html` y `data/db.csv`

In [59]:
df["n_palabras"].describe()

count    335.000000
mean      69.623881
std       89.797787
min        1.000000
25%       26.000000
50%       42.000000
75%       79.500000
max      999.000000
Name: n_palabras, dtype: float64

In [77]:
len(df[df["n_palabras"] > 500])

2

Apenas 2 chunks superan las 500 palabras, y en promedio tienen 69.62 palabras.

In [ ]:
result = client.models.embed_content(
        model="gemini-embedding-2",
        contents="Qué es la UTB"
)
print(len(result.embeddings[0].values))
print(result.embeddings[0].values)

3072
[-0.0084543545, 0.0026820283, 0.017467096, 0.015816968, 0.007845464, 0.005651859, -0.018893791, -0.014389633, -0.014550116, -0.021096813, -0.009959569, -0.0021682666, -0.017655218, 0.0021800192, 0.008722976, 0.020191224, 0.039798453, 0.015661264, -0.0036017296, -0.03072626, 0.013919149, 0.015402151, -0.0039983494, -0.02628925, -0.0029464867, -0.00010449719, 0.0014900429, -0.009309612, 0.010499561, 0.098064125, -0.0036587617, 0.00606777, -0.03372607, 0.011829049, -0.014719766, -0.023166865, 0.019805873, -0.011307208, -0.018213341, -0.03269593, -0.021804638, 0.024836805, -0.02077071, 0.0058645187, 0.006424874, -0.025979225, -0.0063467366, -0.008578566, 0.0058141625, 0.01800149, 0.010773511, 0.009600175, -0.0151689295, 0.0041013253, -0.01463599, -0.0010396262, 0.023640536, 0.019364553, 0.054992057, 0.043057237, -0.018514248, 0.0062520877, -0.0006863256, -0.004035722, 0.005734366, 0.0020182882, 0.029768603, -0.011116853, 0.00611662, 0.016282277, 0.010739885, 0.014312256, 0.023834676, 